In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER
import os

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'BBVA', '2206.pdf')
statement_path

'c:\\Proyectos\\Aether\\Aether_v1\\documents\\inputs\\test_files\\BBVA\\2206.pdf'

In [ ]:
from document_reader import PDFReader

reader = PDFReader(statement_path)
extracted_words = reader.extract_words_from_pdf()

extracted_words.head()

ImportError: attempted relative import with no known parent package

In [3]:
from bank_detector import DefaultBankDetector

bank_detector = DefaultBankDetector(extracted_words)

bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.statement_propertys

print(f'{bank} - {statement_type}')
statement_properties

bbva - debit


{'start_phrase': ['detalle', 'de', 'movimientos', 'realizados'],
 'end_phrase': ['le', 'informamos', 'que', 'puede'],
 'columns': ['OPER',
  'LIQ',
  'DESCRIPCION',
  'REFERENCIA',
  'CARGOS',
  'ABONOS',
  'OPERACION',
  'LIQUIDACION'],
 'date_column': 'OPER',
 'description_column': 'DESCRIPCION',
 'date_pattern': '^\\d{2}/[A-Z]{3}\\b'}

In [4]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table.head()

187 - 886


,page,page_height,text,x0,top,x1,bottom
0,1,792,FECHA,39.098979,516.402066,67.328979,526.402066
1,1,792,SALDO,518.600629,516.402066,547.920629,526.402066
2,1,792,OPER,21.333979,526.402066,43.843979,536.402066
3,1,792,LIQ,70.213979,526.402066,84.213979,536.402066
4,1,792,DESCRIPCION,104.588979,526.402066,161.648979,536.402066


In [5]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table)

grouped_rows = row_segmenter.group_rows()
grouped_rows.head()

,row_group,text,x0,x1,top,bottom,page
0,0,FECHA SALDO,"[39.098978999999886, 518.6006289999999]","[67.32897899999989, 547.9206289999998]",516.402066,526.402066,1
1,1,OPER LIQ DESCRIPCION REFERENCIA CARGOS ABONOS ...,"[21.333978999999886, 70.21397899999988, 104.58...","[43.84397899999989, 84.21397899999988, 161.648...",526.402066,536.402066,1
2,2,15/MAY 16/MAY PAGO CUENTA DE TERCERO 40.00,"[14.803975999999807, 59.42897599999981, 106.83...","[50.373975999999814, 94.99897599999981, 134.29...",543.710649,553.710649,1
3,3,Referencia 1634723960 BNET 1592044105 PAGO PRE...,"[318.5299959999998, 367.10349599999984, 106.17...","[364.4624959999998, 419.92349599999983, 130.18...",555.469940,564.969940,1
4,4,15/MAY 16/MAY SPEI ENVIADO SANTANDER 25.00,"[14.803975999999864, 59.428975999999864, 106.8...","[50.37397599999987, 94.99897599999987, 128.489...",566.229231,576.229231,1


In [6]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, statement_properties)

table_reconstructor.header_row['x0']

[21.333978999999886,
 70.21397899999988,
 104.58897899999988,
 321.1636089999999,
 381.0512709999999,
 423.43566899999985,
 463.42397599999987,
 536.2889759999998]